# Setting up the workspace

## Importing Libraries

In [57]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit

from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

from statsmodels.tsa.stattools import adfuller
import pmdarima as pm

## Setting plot style

In [5]:
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
save_dpi = 150
rf_plot_color = "#009E73"

## Loading and preparing the data

In [7]:
data = pd.read_excel("../data/external/inflation_data.xlsx", sheet_name="Data")
data.columns = [c.strip() for c in data.columns]

In [8]:
# Setting "Date" column type as datetime  
data["Date"] = pd.to_datetime(data["Date"])
data = data.sort_values("Date").reset_index(drop=True)

# Adding time-related features
data["Year"] = data["Date"].dt.year
data["Month"] = data["Date"].dt.month

## Target selection and Test Set size

In [10]:
target_col = "Inflation"
test_size = 12

# Defining functions

## Statistical tests

In [13]:
# Designing ADF test rules
def adf_test(series: pd.Series):
    s = series.dropna()
    if len(s) < 10:
        return np.nan
    try:
        _, pval, *_ = adfuller(s, autolag='AIC')
    except Exception:
        pval = np.nan
    return pval

In [14]:
# Adding interpretations to the ADF test
def interpret_adf(pval, alpha=0.05):
    if np.isnan(pval):
        return "ADF: p=NaN (test failed)"
    return "ADF: Reject unit root (stationary)" if pval < alpha else "ADF: Fail to reject unit root (non-stationary)"

In [15]:
# Defining function to generate stationarity report
def make_stationarity_report(df: pd.DataFrame, diff=False, alpha=0.05) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        if col == "Date" or not np.issubdtype(df[col].dtype, np.number):
            continue
        series = df[col].diff().dropna() if diff else df[col]
        label = "Diff(1)" if diff else "Level"
        adf_p = adf_test(series)
        rows.append({
            "Variable": col,
            "Series": label,
            "ADF_pvalue": adf_p,
            "ADF_interpretation": interpret_adf(adf_p, alpha)
        })
    return pd.DataFrame(rows)

## Reconstructing Inlfation from it's differenciated values

In [17]:
# Designing function to recreate true inflation levels from the differenciated values
def reconstruct_level_from_diff(last_level, diffs):
    out, curr = [], last_level
    for d in np.asarray(diffs).ravel():
        curr += d
        out.append(curr)
    return np.array(out)

In [18]:
#Designing function to recreate true inflation levels from twofold differenciated values
def reconstruct_level_from_diff2(last_level, last_diff, diff2s):
    diff1_preds = reconstruct_level_from_diff(last_diff, diff2s)
    level_preds = reconstruct_level_from_diff(last_level, diff1_preds)
    return level_preds

## MAPE function

In [20]:
# Creating MAPE measure
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    denom = np.where(y_true == 0, 1e-6, np.abs(y_true))
    return np.mean(np.abs((y_true - y_pred) / denom))

# Exploratory Data Analysis

In [51]:
# Selecting numeric columns for further analysis
numeric_level_cols = [c for c in data.columns if c not in ["Date", "Year", "Month"] and np.issubdtype(data[c].dtype, np.number)]
levels_df = data[["Date"] + numeric_level_cols].copy()

# Designing loop to iterate over the features and plot a line plot for each of them
for col in numeric_level_cols:
    plt.figure(figsize=(10, 5))
    plt.plot(levels_df["Date"], levels_df[col], linewidth=2, color='#333333')
    plt.title(f"{col} over time", fontsize=13)
    plt.xlabel("Date")
    plt.ylabel(col)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"../reports/figures/eda_line_{col.replace(' ', '_')}.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()

try:
    pp = sns.pairplot(levels_df[numeric_level_cols].dropna(), corner=True, diag_kind="hist", 
                      plot_kws={'alpha': 0.6, 's': 20})
    pp.fig.suptitle("Pairplot of Economic Indicators", y=1.02, fontsize=14)
    plt.savefig("../reports/figures/eda_pairplot.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()
except Exception:
    pass

# Creating and plotting correlation matrix
corr_mat = levels_df[numeric_level_cols].corr(method='spearman', numeric_only=True)
corr_mat.to_csv("../reports/eda_corr_matrix.csv")
plt.figure(figsize=(10, 8))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="viridis", square=True, 
            cbar_kws={"shrink": 0.8})
plt.title("Correlation Matrix (Levels)", fontsize=14)
plt.tight_layout()
plt.savefig("../reports/figures/eda_corr_heatmap.png", dpi=save_dpi, bbox_inches='tight')
plt.close()

# Modelling

## Data preparation

In [25]:
# Separating raw data into train and test sets
train_raw_idx = len(data) - test_size
train_raw = data.iloc[:train_raw_idx].copy()
test_raw = data.iloc[train_raw_idx:].copy()

# Checking stationarity to adjust the data for RF model (ARIMA does that automatically)
data_rf = data.copy()
rf_diff_orders = {}

print("\n=== Preparing Stationary Data for Random Forest (ADF-based) ===")
cols_to_transform = [c for c in data.columns if c not in ["Date", "Year", "Month"] and np.issubdtype(data[c].dtype, np.number)]

for col in cols_to_transform:
    # Checking stationarity on training data to avoid leakage
    train_series = data_rf[col].iloc[:train_raw_idx].dropna()
    p_val_level = adf_test(train_series)
    
    if p_val_level < 0.05:
        # Stationary at level
        rf_diff_orders[col] = 0
        print(f"Variable '{col}': Level is stationary (ADF p={p_val_level:.4f}). Keeping raw.")
    else:
        # Checking Diff 1
        diff1 = data_rf[col].diff()
        train_series_d1 = diff1.iloc[:train_raw_idx].dropna()
        p_val_d1 = adf_test(train_series_d1)
        
        if p_val_d1 < 0.05:
            rf_diff_orders[col] = 1
            data_rf[col] = diff1
            print(f"Variable '{col}': Diff(1) is stationary (ADF p={p_val_d1:.4f}). Applied Diff(1).")
        else:
            # Applying Diff 2 (will be enoguh for this dataset)
            diff2 = diff1.diff()
            rf_diff_orders[col] = 2
            data_rf[col] = diff2
            train_series_d2 = diff2.iloc[:train_raw_idx].dropna()
            p_val_d2 = adf_test(train_series_d2)
            print(f"Variable '{col}': Diff(1) non-stationary. Applied Diff(2) (ADF p={p_val_d2:.4f}).")

# Split RF Data (Dropping NaNs created by differencing
train_rf = data_rf.iloc[:train_raw_idx].dropna().copy()
test_rf = data_rf.iloc[train_raw_idx:].dropna().copy()

# Need to keep track of last values for reconstruction if target was differenced
target_diff_order = rf_diff_orders.get(target_col, 0)
last_train_level = train_raw[target_col].iloc[-1]
last_train_diff1 = None
if target_diff_order >= 2:
    temp_diff1 = train_raw[target_col].diff()
    last_train_diff1 = temp_diff1.iloc[-1]


=== Preparing Stationary Data for Random Forest (ADF-based) ===
Variable 'Inflation': Diff(1) is stationary (ADF p=0.0002). Applied Diff(1).
Variable 'Interest Rate': Diff(1) is stationary (ADF p=0.0003). Applied Diff(1).
Variable 'M2': Diff(1) non-stationary. Applied Diff(2) (ADF p=0.0000).
Variable 'Industry Production': Level is stationary (ADF p=0.0017). Keeping raw.
Variable 'Import': Diff(1) is stationary (ADF p=0.0069). Applied Diff(1).
Variable 'Export': Diff(1) is stationary (ADF p=0.0010). Applied Diff(1).
Variable 'USDPLN': Diff(1) is stationary (ADF p=0.0000). Applied Diff(1).
Variable 'USD Reserves': Diff(1) is stationary (ADF p=0.0002). Applied Diff(1).


## Fitting the models

In [27]:
# Defining feature configurations for different models
base_vars = ["Interest Rate", "M2", "Import", "Export", "Industry Production"]
datasets_config = {
    "data_clean_1": base_vars,
    "data_clean_2": base_vars + ["USDPLN"],
    "data_clean_3": base_vars + ["USDPLN", "USD Reserves"]
}

results_all = []
y_level_truth = test_raw[target_col].values

# Storage for Combined  Plot
rf_importance_data = {}

# Generating stationarity stats for report
adf_level_train = adf_test(train_raw[target_col])
adf_diff_train = adf_test(train_raw[target_col].diff())

In [28]:
# Modelling pipeline
for name, vars_list in datasets_config.items():
    print(f"\n=== Running {name}, test size = {test_size} ===")

    vars_list_with_time = vars_list + ["Year", "Month"]
    models_preds_level = {}
    
    # === PATH A: Raw Data (Linear Regression, ARMA, ARIMAX) ===
    x_train_raw = train_raw[vars_list_with_time]
    y_train_raw = train_raw[target_col]
    x_test_raw = test_raw[vars_list_with_time]
    
    # 1. Linear Regression
    lr = Pipeline([("scaler", StandardScaler()), ("lr", LinearRegression())])
    lr.fit(x_train_raw, y_train_raw)
    models_preds_level["Linear Regression"] = lr.predict(x_test_raw)

    # 2. ARMA
    try:
        arma = pm.auto_arima(y_train_raw, X=x_train_raw, seasonal=False, d=0, max_d=0,
                                stepwise=True, suppress_warnings=True)
        arma_pred = arma.predict(n_periods=len(x_test_raw), X=x_test_raw)
        print(f"  >> [ARMA] Best Order (p,d,q): {arma.order}")
        
    except Exception:
        arma_pred = np.full(len(x_test_raw), np.nan)
        print("  >> [ARMA] Failed to fit.")
    models_preds_level["ARMA (d=0)"] = arma_pred
    
    # 3. Auto ARIMAX
    try:
        arimax = pm.auto_arima(y_train_raw, X=x_train_raw, 
                               test='adf',
                               seasonal=False, 
                               stepwise=True, suppress_warnings=True)
        arimax_pred = arimax.predict(n_periods=len(x_test_raw), X=x_test_raw)
        print(f"  >> [Auto ARIMAX] Best Order (p,d,q): {arimax.order}")
        
    except Exception:
        arimax_pred = np.full(len(x_test_raw), np.nan)
        print("  >> [Auto ARIMAX] Failed to fit.")
    models_preds_level["Auto ARIMAX"] = arimax_pred
    
    # === PATH B: Stationary Data (Random Forest) ===
    # 4. Random Forest
    valid_rf_vars = [v for v in vars_list_with_time if v in train_rf.columns]
    
    x_train_rf = train_rf[valid_rf_vars]
    y_train_rf = train_rf[target_col]
    x_test_rf = test_rf[valid_rf_vars]

    tscv = TimeSeriesSplit(n_splits=5)
    rf_search = BayesSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        search_spaces={
            'n_estimators': Integer(200, 1000),
            'max_depth': Integer(3, 30),
            'min_samples_split': Integer(2, 20),
            'min_samples_leaf': Integer(1, 20),
            'max_features': Categorical(['sqrt', 'log2'])
        },
        n_iter=50,
        cv=tscv,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        verbose=0,
        random_state=42
    )
    rf_search.fit(x_train_rf, y_train_rf)
    best_rf = rf_search.best_estimator_
    
    print(f"  >> [Random Forest] Best Params: n_estimators={best_rf.n_estimators}, "
          f"max_depth={best_rf.max_depth}, min_samples_split={best_rf.min_samples_split}")
    
    rf_pred_stat = best_rf.predict(x_test_rf)
    
    if target_diff_order == 0:
        models_preds_level["Random Forest (Tuned)"] = rf_pred_stat
    elif target_diff_order == 1:
        models_preds_level["Random Forest (Tuned)"] = reconstruct_level_from_diff(last_train_level, rf_pred_stat)
    elif target_diff_order == 2:
        models_preds_level["Random Forest (Tuned)"] = reconstruct_level_from_diff2(last_train_level, last_train_diff1, rf_pred_stat)
    
    rf_len = len(models_preds_level["Random Forest (Tuned)"])
    target_len = len(y_level_truth)
    if rf_len > target_len:
        models_preds_level["Random Forest (Tuned)"] = models_preds_level["Random Forest (Tuned)"][:target_len]

    # Evaluation & Aggregation
    min_len = min(len(y_level_truth), *[len(v) for v in models_preds_level.values()])
    inflation_test_aligned = y_level_truth[:min_len]
    models_preds_aligned = {k: v[:min_len] for k, v in models_preds_level.items()}

    # Collecting data for plots
    if hasattr(best_rf, "feature_importances_"):
        imp_df = pd.DataFrame({
            "Feature": x_train_rf.columns,
            "Importance": best_rf.feature_importances_
        }).sort_values("Importance", ascending=False).head(10)
        rf_importance_data[name] = imp_df


    scenario_results = []
    for model_name, y_pred_level in models_preds_aligned.items():
        y_true_level = inflation_test_aligned
        rmse = float(np.sqrt(mean_squared_error(y_true_level, y_pred_level)))
        mae = float(mean_absolute_error(y_true_level, y_pred_level))
        mape_score = float(mape(y_true_level, y_pred_level))
        scenario_results.append({
            "Dataset": name,
            "Test_Size": test_size,
            "Model": model_name,
            "MAE_level": mae,
            "RMSE_level": rmse,
            "MAPE_level": mape_score
        })

    df_results = pd.DataFrame(scenario_results)
    results_all.append(df_results)


=== Running data_clean_1, test size = 12 ===
  >> [ARMA] Best Order (p,d,q): (1, 0, 0)
  >> [Auto ARIMAX] Best Order (p,d,q): (1, 1, 1)
  >> [Random Forest] Best Params: n_estimators=200, max_depth=28, min_samples_split=20

=== Running data_clean_2, test size = 12 ===
  >> [ARMA] Best Order (p,d,q): (1, 0, 0)
  >> [Auto ARIMAX] Best Order (p,d,q): (1, 1, 1)
  >> [Random Forest] Best Params: n_estimators=200, max_depth=3, min_samples_split=20

=== Running data_clean_3, test size = 12 ===
  >> [ARMA] Best Order (p,d,q): (1, 0, 0)
  >> [Auto ARIMAX] Best Order (p,d,q): (1, 1, 1)
  >> [Random Forest] Best Params: n_estimators=200, max_depth=30, min_samples_split=20


# Generating reports

In [30]:
# Saving stationarity reports
stationarity_levels_df = make_stationarity_report(levels_df, diff=False)
stationarity_diffs_df = make_stationarity_report(levels_df, diff=True)
stationarity_levels_df.to_csv("../reports/stationarity_report_levels.csv", index=False)
stationarity_diffs_df.to_csv("../reports/stationarity_report_diffs.csv", index=False)

In [31]:
# Plotting combined feature importance
if len(rf_importance_data) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), sharey=False) 
    
    plot_order = ["data_clean_1", "data_clean_2", "data_clean_3"]
    plot_titles = ["Base Features", "Base + USDPLN", "Base + USDPLN + USD Reserves"]

    for ax, key, title in zip(axes, plot_order, plot_titles):
        if key not in rf_importance_data:
            continue
        df_imp = rf_importance_data[key]
        
        sns.barplot(data=df_imp, x="Importance", y="Feature", ax=ax, 
                    color=rf_plot_color, orient='h', alpha=0.8)
        
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel("Importance Score", fontsize=10)
        ax.set_ylabel("") # Remove y-label to save space
        
    plt.tight_layout()
    plt.savefig("../reports/figures/rf_feature_importance_comparison.png", dpi=save_dpi, bbox_inches='tight')
    plt.close()

In [32]:
# Generating error sumary table
errors_summary_all = pd.concat(results_all, axis=0, ignore_index=True)
errors_summary_all.to_csv("../reports/errors_summary_all.csv", index=False)